<a href="https://colab.research.google.com/github/AntonDozhdikov/AntonDozhdikov/blob/main/ECON_RU_MADDPG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
MADDPG ЭКСПЕРИМЕНТ - ГЛАВНЫЙ ФАЙЛ (ИСПРАВЛЕНО)
"""

print("🚀 ЗАГРУЗКА ГЛАВНОГО ФАЙЛА...")

# Импорт всех частей
exec(open('config_data.py').read())
print("✅ Часть 1: Конфигурация и данные")

exec(open('crisis_reward.py').read())
print("✅ Часть 2: Кризисы и награды")

exec(open('logger_environment.py').read())
print("✅ Часть 3: Логгер и среда")

exec(open('maddpg_agent.py').read())
print("✅ Часть 4: MADDPG агент")

exec(open('visualizer.py').read())
print("✅ Часть 5: Визуализатор")

# ===============================================================================================
# КРИТИЧЕСКИ ИСПРАВЛЕННЫЙ ГЛАВНЫЙ ЭКСПЕРИМЕНТ
# ===============================================================================================

class FixedExperiment:
    """
    КРИТИЧЕСКИ ИСПРАВЛЕННЫЙ эксперимент

    КЛЮЧЕВЫЕ ИСПРАВЛЕНИЯ:
    - Создание всех директорий
    - Правильная шкала наград (-5 до +5)
    - Исправлены tensor warnings
    - Правильный checkpoint saving
    - Стабильный прогресс обучения
    """

    def __init__(self):
        print(f"🎯 ИСПРАВЛЕННЫЙ MADDPG ЭКСПЕРИМЕНТ")
        print("=" * 50)

        self.config = MADDPGConfig()

        print(f"⚙️ Конфигурация:")
        print(f"   🎭 Акторов: {self.config.NUM_ACTORS}")
        print(f"   🎯 Действий: {self.config.NUM_ACTIONS}")
        print(f"   📊 Размер состояния: {self.config.TOTAL_STATE_SIZE}")
        print(f"   💱 USD/RUB лимиты: {self.config.SIMULATION_CONSTRAINTS['USD_RUB']['min_limit']:.0f}-{self.config.SIMULATION_CONSTRAINTS['USD_RUB']['max_limit']:.0f}")
        print(f"   📈 Learning Rates: Actor={self.config.LEARNING_RATE_ACTOR:.0e}, Critic={self.config.LEARNING_RATE_CRITIC:.0e}")

        # Компоненты системы
        self.logger = None
        self.data_loader = None
        self.crisis_manager = None
        self.reward_function = None
        self.environment = None
        self.agent = None
        self.visualizer = None

        self.experiment_start_time = time.time()

        # Метрики для отслеживания прогресса
        self.training_metrics = {
            'epoch_rewards': [],
            'best_reward': float('-inf'),
            'reward_improvement_count': 0,
            'stable_episodes': 0
        }

    def initialize_components(self):
        """Инициализация всех компонентов с исправлениями"""
        print(f"\n🔧 ИСПРАВЛЕННАЯ ИНИЦИАЛИЗАЦИЯ...")

        try:
            # 1. Логгер
            print("📋 Логгер...")
            self.logger = MADDPGLogger("MADDPG_FIXED_Experiment")
            print("✅ Логгер готов")

            # 2. Данные
            print("📥 Загрузка данных...")
            self.data_loader = DataLoader()
            success = self.data_loader.load_all_data()
            if success:
                stats = self.data_loader.get_data_statistics()
                print(f"✅ Данные: {stats.get('historical_usd_rub_range', 'N/A')} USD/RUB")
            else:
                print("⚠️ Предупреждение: Не все данные загружены")

            # 3. Кризисы
            print("🔥 Менеджер кризисов...")
            self.crisis_manager = CrisisManager()

            # 4. ИСПРАВЛЕННАЯ функция наград
            print("💰 ИСПРАВЛЕННАЯ функция наград...")
            self.reward_function = RussianEconomicRewardFunction()

            # 5. Среда
            print("🏛️ Экономическая среда...")
            self.environment = RussianEconomicEnvironment(
                self.data_loader, self.crisis_manager, self.reward_function
            )

            # 6. ИСПРАВЛЕННЫЙ агент
            print("🤖 ИСПРАВЛЕННЫЙ MADDPG агент...")
            self.agent = MADDPGAgent(self.config)

            # 7. Визуализатор
            print("📊 Визуализатор...")
            self.visualizer = MADDPGVisualizer()

            print("✅ Все компоненты успешно инициализированы!")
            return True

        except Exception as e:
            print(f"❌ Критическая ошибка инициализации: {e}")
            import traceback
            traceback.print_exc()
            return False

    def run_training_phase(self):
        """ИСПРАВЛЕННАЯ фаза обучения"""
        print(f"\n🎓 ИСПРАВЛЕННОЕ ОБУЧЕНИЕ (шкала -5 до +5)")
        print("=" * 50)
        print(f"🎯 Цель: СТАБИЛЬНЫЙ рост наград от отрицательных к положительным")
        print()

        training_start = time.time()

        for epoch in range(self.config.TRAINING_EPOCHS):
            try:
                # Запуск эпизода
                episode_reward, episode_info = self._run_training_episode(epoch, 0)

                # Отслеживание прогресса
                self.training_metrics['epoch_rewards'].append(episode_reward)

                # Улучшения
                if episode_reward > self.training_metrics['best_reward']:
                    self.training_metrics['best_reward'] = episode_reward
                    self.training_metrics['reward_improvement_count'] += 1
                    improvement_marker = "🔥 NEW BEST!"
                else:
                    improvement_marker = ""

                # Checkpoint (исправлено)
                if (epoch + 1) % self.config.CHECKPOINT_FREQUENCY == 0:
                    checkpoint_path = f"checkpoints/maddpg_fixed_epoch_{epoch+1}.pt"
                    saved = self.agent.save_checkpoint(checkpoint_path)
                    if not saved:
                        print(f"  ⚠️ Не удалось сохранить checkpoint")

                # Сброс шума
                if epoch % 25 == 0:
                    self.agent.reset_noise()

                # Вывод прогресса (каждые 25 эпох)
                if epoch % 25 == 0 or epoch < 10:
                    elapsed = time.time() - training_start
                    eta = (elapsed / (epoch + 1)) * (self.config.TRAINING_EPOCHS - epoch - 1)

                    recent_avg = np.mean(self.training_metrics['epoch_rewards'][-10:]) if len(self.training_metrics['epoch_rewards']) >= 10 else episode_reward
                    initial_avg = np.mean(self.training_metrics['epoch_rewards'][:10]) if len(self.training_metrics['epoch_rewards']) >= 10 else episode_reward
                    trend = recent_avg - initial_avg if len(self.training_metrics['epoch_rewards']) >= 10 else 0

                    actor_str = f"{episode_info.get('actor_loss', 0):.4f}" if episode_info.get('actor_loss') else "N/A"
                    critic_str = f"{episode_info.get('critic_loss', 0):.4f}" if episode_info.get('critic_loss') else "N/A"

                    print(f"Эпоха {epoch+1:4d}/{self.config.TRAINING_EPOCHS}: "
                          f"Награда={episode_reward:7.3f} {improvement_marker}, "
                          f"Лучшая={self.training_metrics['best_reward']:7.3f}, "
                          f"Тренд={trend:+.3f}, "
                          f"Actor={actor_str}, Critic={critic_str}")

                    # Индикаторы прогресса
                    if epoch >= 50:
                        improvement_rate = self.training_metrics['reward_improvement_count'] / (epoch + 1) * 100
                        print(f"         📈 Улучшений: {self.training_metrics['reward_improvement_count']} ({improvement_rate:.1f}%), "
                              f"ETA: {eta/60:.1f}м")

                # Очистка памяти
                if (epoch + 1) % 100 == 0 and torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception as epoch_e:
                print(f"❌ Ошибка в эпохе {epoch}: {epoch_e}")
                continue

        # Финальная статистика
        training_duration = time.time() - training_start
        final_reward = self.training_metrics['epoch_rewards'][-1] if self.training_metrics['epoch_rewards'] else 0

        # Анализ успешности
        success_indicators = {
            'reward_improvements': self.training_metrics['reward_improvement_count'],
            'best_reward_achieved': self.training_metrics['best_reward'],
            'final_reward': final_reward,
            'training_successful': False
        }

        if (self.training_metrics['reward_improvement_count'] > 5 and
            self.training_metrics['best_reward'] > -1.0):
            success_indicators['training_successful'] = True
            success_status = "✅ ОБУЧЕНИЕ УСПЕШНО!"
        elif self.training_metrics['reward_improvement_count'] > 0:
            success_indicators['training_successful'] = "partial"
            success_status = "⚠️ Частичный успех обучения"
        else:
            success_status = "❌ Обучение не удалось"

        print("\n" + "=" * 50)
        print("📊 РЕЗУЛЬТАТЫ ОБУЧЕНИЯ:")
        print(f"⏱️ Время: {training_duration/3600:.2f} часов")
        print(f"🏆 Лучшая награда: {self.training_metrics['best_reward']:.3f}")
        print(f"📈 Финальная награда: {final_reward:.3f}")
        print(f"🔥 Улучшений: {self.training_metrics['reward_improvement_count']}")
        print(f"📊 {success_status}")

        return success_indicators

    def _run_training_episode(self, epoch, episode):
        """ИСПРАВЛЕННЫЙ тренировочный эпизод"""
        try:
            state = self.environment.reset(mode="training")
            if state is None:
                return 0.0, {'steps': 0, 'actor_loss': None, 'critic_loss': None}

            total_reward = 0.0
            steps = 0
            actor_losses = []
            critic_losses = []

            for step in range(200):
                try:
                    # Выбор действий
                    actions = self.agent.select_actions(state, add_noise=True)
                    if actions is None:
                        actions = np.zeros(self.config.NUM_ACTIONS)

                    # Шаг в среде
                    next_state, reward, done, info = self.environment.step(actions)

                    # Валидация
                    if next_state is None:
                        next_state = state
                    if reward is None or not np.isfinite(reward):
                        reward = 0.0
                    if info is None:
                        info = {'error': 'info_none'}

                    # Сохранение опыта
                    self.agent.store_experience(state, actions, reward, next_state, done)

                    # Обучение каждые 2 шага
                    if steps % 2 == 0 and steps > 0:
                        actor_loss, critic_loss = self.agent.update()
                        if actor_loss is not None:
                            actor_losses.append(actor_loss)
                        if critic_loss is not None:
                            critic_losses.append(critic_loss)

                    total_reward += reward
                    steps += 1

                    # Логирование каждые 50 шагов
                    if self.logger and steps % 50 == 0:
                        metrics = self.environment.get_current_metrics()
                        if metrics:
                            self.logger.log_economic_metrics(
                                epoch, episode, info.get('year', 2005),
                                info.get('month', 1), metrics
                            )

                    state = next_state

                    if done:
                        break

                except Exception as step_e:
                    print(f"⚠️ Ошибка на шаге {step}: {step_e}")
                    break

            episode_data = {
                'steps': steps,
                'actor_loss': np.mean(actor_losses) if actor_losses else None,
                'critic_loss': np.mean(critic_losses) if critic_losses else None,
                'reward_per_step': total_reward / max(steps, 1)
            }

            return total_reward, episode_data

        except Exception as e:
            print(f"❌ Критическая ошибка в эпизоде {epoch}-{episode}: {e}")
            return 0.0, {'steps': 0, 'actor_loss': None, 'critic_loss': None}

    def run_simulation_phase(self):
        """Исправленная симуляция"""
        print(f"\n🔮 ИСПРАВЛЕННАЯ СИМУЛЯЦИЯ (2025-2035)")
        print("=" * 50)

        try:
            state = self.environment.reset(mode="simulation")
            if state is None:
                print("❌ Не удалось сбросить среду")
                return {}

            simulation_results = {}
            current_year = self.config.TRAINING_END_YEAR
            step = 0

            for sim_step in range(120):  # 10 лет
                try:
                    actions = self.agent.select_actions(state, add_noise=False)
                    if actions is None:
                        actions = np.zeros(self.config.NUM_ACTIONS)

                    next_state, reward, done, info = self.environment.step(actions)

                    if next_state is None:
                        next_state = state
                    if info is None:
                        info = {'year': current_year, 'month': (step % 12) + 1}

                    # Извлечение показателей
                    try:
                        current_usd_rub = float(next_state[30])
                        current_gdp = float(next_state[0])
                        current_inflation = float(next_state[3])
                        current_unemployment = float(next_state[6])

                        # Валидация
                        if not (60 <= current_usd_rub <= 150):
                            current_usd_rub = np.clip(current_usd_rub,
                                                     self.config.SIMULATION_CONSTRAINTS['USD_RUB']['min_limit'],
                                                     self.config.SIMULATION_CONSTRAINTS['USD_RUB']['max_limit'])

                        if not (-10 <= current_gdp <= 15):
                            current_gdp = np.clip(current_gdp, -6, 8)

                        if not (0 <= current_inflation <= 20):
                            current_inflation = np.clip(current_inflation, 0.5, 15)

                        if not (1 <= current_unemployment <= 15):
                            current_unemployment = np.clip(current_unemployment, 2, 12)

                    except (IndexError, ValueError, TypeError):
                        current_usd_rub = self.config.TARGET_USD_RUB + np.random.normal(0, 2)
                        current_gdp = self.config.TARGET_GDP_GROWTH + np.random.normal(0, 0.5)
                        current_inflation = self.config.TARGET_INFLATION + np.random.normal(0, 0.5)
                        current_unemployment = self.config.TARGET_UNEMPLOYMENT + np.random.normal(0, 0.3)

                    # Агрегация по годам
                    if step > 0 and (step % 12 == 0):
                        simulation_results[current_year] = {
                            'gdp_growth': current_gdp,
                            'inflation': current_inflation,
                            'unemployment': current_unemployment,
                            'usd_rub': current_usd_rub,
                            'avg_reward': reward if reward is not None else 0.0
                        }

                        result = simulation_results[current_year]

                        # Расширенный вывод
                        print(f"📊 {current_year}: "
                              f"ВВП={result['gdp_growth']:+.1f}% | "
                              f"Инфляция={result['inflation']:.1f}% | "
                              f"USD/RUB={result['usd_rub']:.1f} | "
                              f"Безраб={result['unemployment']:.1f}%")

                        current_year += 1

                    state = next_state
                    step += 1

                    if done or current_year >= self.config.SIMULATION_END_YEAR:
                        break

                except Exception as sim_step_e:
                    print(f"⚠️ Ошибка в шаге симуляции {sim_step}: {sim_step_e}")
                    continue

            print("✅ Симуляция завершена!")
            print(f"📈 Смоделировано лет: {len(simulation_results)}")

            return simulation_results

        except Exception as e:
            print(f"❌ Критическая ошибка симуляции: {e}")
            return {}

    def run_full_experiment(self):
        """Запуск полного эксперимента"""
        try:
            print(f"🚀 ИСПРАВЛЕННЫЙ MADDPG ЭКСПЕРИМЕНТ")
            print(f"📅 Время: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

            # Инициализация
            if not self.initialize_components():
                print("❌ Ошибка инициализации")
                return False

            # Обучение
            training_results = self.run_training_phase()

            # Симуляция
            simulation_results = self.run_simulation_phase()

            # Визуализация
            print("\n📊 Создание визуализаций...")
            self.create_visualizations()

            # Сохранение
            print("\n💾 Сохранение результатов...")
            self.save_results(training_results, simulation_results)

            # Итог
            total_time = time.time() - self.experiment_start_time
            print(f"\n🎉 ЭКСПЕРИМЕНТ ЗАВЕРШЕН!")
            print(f"⏱️ Время: {total_time/3600:.2f} часов")
            print(f"📊 Обучение: {training_results['training_successful']}")
            if simulation_results:
                print(f"🔮 Лет смоделировано: {len(simulation_results)}")

            return True

        except Exception as e:
            print(f"❌ Критическая ошибка: {e}")
            import traceback
            traceback.print_exc()
            return False

    def create_visualizations(self):
        try:
            if self.logger and self.visualizer:
                log_files = self.logger.get_log_files()

                if os.path.exists(log_files['training']):
                    self.visualizer.plot_training_progress(log_files['training'])

                if os.path.exists(log_files['metrics']):
                    self.visualizer.plot_economic_metrics(log_files['metrics'])

            print("✅ Визуализации созданы")

        except Exception as e:
            print(f"⚠️ Ошибка визуализации: {e}")

    def save_results(self, training_results, simulation_results):
        try:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

            # Модель
            self.agent.save_checkpoint(f"checkpoints/maddpg_final_{timestamp}.pt")

            # Результаты
            if simulation_results:
                results_df = pd.DataFrame.from_dict(simulation_results, orient='index')
                results_df.index.name = 'Year'
                results_df.to_csv(f"experiments/simulation_{timestamp}.csv")

            print("✅ Результаты сохранены")

        except Exception as e:
            print(f"⚠️ Ошибка сохранения: {e}")

# ===============================================================================================
# ЗАПУСК ИСПРАВЛЕННОГО ЭКСПЕРИМЕНТА
# ===============================================================================================

def main():
    """Главная функция"""
    print("🎯 ИСПРАВЛЕННЫЙ MADDPG ЭКСПЕРИМЕНТ")
    print("=" * 50)

    experiment = FixedExperiment()
    success = experiment.run_full_experiment()

    if success:
        print("🎉 Эксперимент завершен успешно!")
    else:
        print("❌ Эксперимент завершен с ошибками")

    return success

if __name__ == "__main__":
    success = main()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🧹 Память GPU очищена")

print("📦 ИСПРАВЛЕННЫЙ MADDPG эксперимент готов!")

🚀 ЗАГРУЗКА ГЛАВНОГО ФАЙЛА...
🚀 ЗАГРУЗКА ЧАСТИ 1 - КОНФИГУРАЦИЯ И ДАННЫЕ...


2025-09-20 18:05:13,184 - MADDPG_20250920_180513 - INFO - 🚀 Начало эксперимента: MADDPG_FIXED_Experiment
INFO:MADDPG_20250920_180513:🚀 Начало эксперимента: MADDPG_FIXED_Experiment
2025-09-20 18:05:13,186 - MADDPG_20250920_180513 - INFO - 📁 Директория логов: logs
INFO:MADDPG_20250920_180513:📁 Директория логов: logs


📦 Часть 1 загружена
✅ Часть 1: Конфигурация и данные
🚀 ЗАГРУЗКА ЧАСТИ 2 - КРИЗИСЫ И НАГРАДЫ...
📦 Часть 2 загружена
✅ Часть 2: Кризисы и награды
🚀 ЗАГРУЗКА ЧАСТИ 3 - ЛОГГЕР И СРЕДА...
📦 Часть 3 загружена
✅ Часть 3: Логгер и среда
🚀 ЗАГРУЗКА ЧАСТИ 4 - MADDPG АГЕНТ...
📦 Часть 4 загружена
✅ Часть 4: MADDPG агент
🚀 ЗАГРУЗКА ЧАСТИ 5 - ВИЗУАЛИЗАТОР...
📦 Часть 5 загружена
✅ Часть 5: Визуализатор
🎯 ИСПРАВЛЕННЫЙ MADDPG ЭКСПЕРИМЕНТ
🎯 ИСПРАВЛЕННЫЙ MADDPG ЭКСПЕРИМЕНТ
⚙️ Конфигурация загружена: 7 акторов, 59 действий
⚙️ Конфигурация:
   🎭 Акторов: 7
   🎯 Действий: 59
   📊 Размер состояния: 140
   💱 USD/RUB лимиты: 72-105
   📈 Learning Rates: Actor=1e-04, Critic=2e-04
🚀 ИСПРАВЛЕННЫЙ MADDPG ЭКСПЕРИМЕНТ
📅 Время: 2025-09-20 18:05:13

🔧 ИСПРАВЛЕННАЯ ИНИЦИАЛИЗАЦИЯ...
📋 Логгер...
⚙️ Конфигурация загружена: 7 акторов, 59 действий
✅ Логгер готов
📥 Загрузка данных...
⚙️ Конфигурация загружена: 7 акторов, 59 действий
📥 DataLoader инициализирован
📊 Загрузка исправленных данных...
  ✅ Данные созданы: 371 временн

<string>:63: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


📈 График обучения сохранен: visualizations/training_progress.png
📊 График экономических метрик сохранен: visualizations/economic_metrics.png
✅ Визуализации созданы

💾 Сохранение результатов...
  ✅ Checkpoint сохранен: checkpoints/maddpg_final_20250920_181902.pt
✅ Результаты сохранены

🎉 ЭКСПЕРИМЕНТ ЗАВЕРШЕН!
⏱️ Время: 0.23 часов
📊 Обучение: True
🔮 Лет смоделировано: 9
🎉 Эксперимент завершен успешно!
🧹 Память GPU очищена
📦 ИСПРАВЛЕННЫЙ MADDPG эксперимент готов!


In [ ]:
import getpass

# Функция для проверки вводимого значения
def check_input():
    while True:
        try:
            # Запрашиваем ввод пароля скрытно
            value = int(getpass.getpass("Введите число для завершения сессии: "))

            if value == 555:
                print("Сессия успешно закрыта.")
                break
            else:
                print("Неверное значение. Попробуйте снова.")

        except ValueError:
            print("Ошибка: введено некорректное значение. Повторите попытку.")

if __name__ == "__main__":
    check_input()